## Query

In [ ]:
import os
import pandas as pd
print(f"Current working directory: {os.getcwd()}")

In [31]:
import pybliometrics
pybliometrics.scopus.init()

In [35]:
# Scopus
from pybliometrics.scopus import ScopusSearch

# Define your query (same syntax as the Scopus search bar)
query = 'TITLE-ABS-KEY("hybrid modeling" OR "hybrid modelling" OR "physics-informed machine learning" OR "grey box model" OR "gray box model")'

# Perform the search
s = ScopusSearch(query, verbose=True)

# Convert the results to a list of dictionaries or a DataFrame
results_df = pd.DataFrame(s.results)

print(f"Found {len(results_df)} papers.")
results_df[['title', 'publicationName', 'doi']].head()

100%|██████████| 388/388 [09:11<00:00,  1.43s/it]


Found 9700 papers.


,title,publicationName,doi
0,ARIMA-based forecasting of cerebral physiologi...,Intensive Care Medicine Experimental,10.1186/s40635-026-00855-y
1,Machine learning based variance estimation und...,Scientific Reports,10.1038/s41598-026-36844-0
2,Laser surface modification of additively manuf...,Journal of Engineering and Applied Science,10.1186/s44147-026-00936-5
3,A conceptual model for machine learning based ...,Discover Artificial Intelligence,10.1007/s44163-026-00946-5
4,Integrating machine learning and physics-based...,Scientific Reports,10.1038/s41598-026-37098-6


## Analyze query

In [15]:
import pandas as pd

In [16]:
ref_files = [
    "../data/baselines/00 Perspectives on the integration between first-principles and data-driven modeling.csv",
    "../data/baselines/01 Physics-informed machine learning _ A comprehensive review on applications in anomaly detection and condition monitoring.csv"
]

In [17]:
example_df = pd.read_csv(ref_files[0], encoding="utf-8", header=0)
example_df.head()

,Authors,Author full names,Author(s) ID,Title,Year,Source title,Volume,Issue,Art. No.,Page start,Page end,Cited by,DOI,Link,Document Type,Publication Stage,Open Access,Source,EID
0,Bangi M.S.F.; Kwon J.S.-I.,"Bangi, Mohammed Saad Faizan (57208246818); Kwo...",57208246818; 55256605700,Deep hybrid modeling of chemical process: Appl...,2020,Computers and Chemical Engineering,134,NaN,106696,NaN,NaN,145,10.1016/j.compchemeng.2019.106696,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,All Open Access; Green Open Access,Scopus,2-s2.0-85077644754
1,Wang X.; Chen J.; Liu C.; Pan F.,"Wang, Xianfang (24081857500); Chen, Jindong (3...",24081857500; 35387691100; 55680771900; 5720148...,Hybrid modeling of penicillin fermentation pro...,2010,Chemical Engineering Research and Design,88,4,NaN,415,420,75,10.1016/j.cherd.2009.08.010,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,NaN,Scopus,2-s2.0-77951296145
2,Venkatasubramanian V.,"Venkatasubramanian, Venkat (7006834244)",7006834244,The promise of artificial intelligence in chem...,2019,AIChE Journal,65,2,NaN,466,478,557,10.1002/aic.16489,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,NaN,Scopus,2-s2.0-85058687299
3,Linkletter C.; Bingham D.; Hengartner N.; Higd...,"Linkletter, Crystal (35291158200); Bingham, De...",35291158200; 57204216471; 6603808707; 66040758...,Variable selection for Gaussian process models...,2006,Technometrics,48,4,NaN,478,490,127,10.1198/004017006000000228,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,NaN,Scopus,2-s2.0-33845260356
4,Wang S.; Yu X.; Perdikaris P.,"Wang, Sifan (57219690904); Yu, Xinling (578085...",57219690904; 57808523900; 35194560500,When and why PINNs fail to train: A neural tan...,2022,Journal of Computational Physics,449,NaN,110768,NaN,NaN,970,10.1016/j.jcp.2021.110768,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,All Open Access; Green Open Access,Scopus,2-s2.0-85118866669


In [44]:
ref_doi = set()
total_doi = 0
ref_doi_dfs = []
verbose = False
for file in ref_files:
    df = pd.read_csv(file, encoding="utf-8", header=0)
    file_doi = df["DOI"].dropna().unique()
    ref_doi_dfs.append(df)
    total_doi += len(file_doi)
    print(f"-> {len(file_doi)} entries for '{file}'")
    ref_doi.update(file_doi)
print(f"Total unique DOIs: {len(ref_doi)}")
print(f"Total DOIs across all files: {total_doi}")
ref_doi_df = pd.concat(ref_doi_dfs, ignore_index=True)
#remove rows with nan in the doi column
ref_doi_df = ref_doi_df.dropna(subset=["DOI"])

-> 142 entries for '../data/baselines/00 Perspectives on the integration between first-principles and data-driven modeling.csv'
-> 132 entries for '../data/baselines/01 Physics-informed machine learning _ A comprehensive review on applications in anomaly detection and condition monitoring.csv'
Total unique DOIs: 268
Total DOIs across all files: 274


In [6]:
from pybliometrics.scopus import ScopusSearch
# Your set of 268 DOIs
doi_list = list(ref_doi)

chunk_size = 25
results = []

for i in range(0, len(doi_list), chunk_size):
    # 1. Create a sub-list of DOIs
    chunk = doi_list[i : i + chunk_size]

    # 2. Build the Scopus query string: DOI("...") OR DOI("...")
    query = " OR ".join([f'DOI("{d}")' for d in chunk])

    try:
        # 3. Use ScopusSearch with the COMPLETE view to get abstracts
        # This reduces 268 individual calls to roughly 11 batch calls
        search_res = ScopusSearch(query, view="COMPLETE")

        # 4. Extract data from the search result object
        if search_res.results:
            for doc in search_res.results:
                results.append({
                    "doi": doc.doi,
                    "title": doc.title,
                    "abstract": doc.description, # 'description' contains the abstract in Search API
                    "journal": doc.publicationName,
                    "date": doc.coverDate
                })
        print(f"Processed chunk {i // chunk_size + 1}: {len(chunk)} DOIs")

    except Exception as e:
        print(f"Error in batch {i // chunk_size + 1}: {e}")

# Save to CSV for analysis
df = pd.DataFrame(results)

Error in batch 1: No configuration file found.Please initialize Pybliometrics with init().
For more information visit: https://pybliometrics.readthedocs.io/en/stable/configuration.html
Error in batch 2: No configuration file found.Please initialize Pybliometrics with init().
For more information visit: https://pybliometrics.readthedocs.io/en/stable/configuration.html
Error in batch 3: No configuration file found.Please initialize Pybliometrics with init().
For more information visit: https://pybliometrics.readthedocs.io/en/stable/configuration.html
Error in batch 4: No configuration file found.Please initialize Pybliometrics with init().
For more information visit: https://pybliometrics.readthedocs.io/en/stable/configuration.html
Error in batch 5: No configuration file found.Please initialize Pybliometrics with init().
For more information visit: https://pybliometrics.readthedocs.io/en/stable/configuration.html
Error in batch 6: No configuration file found.Please initialize Pybliometri

In [10]:
df.to_csv("../data/baselines/merged_metadata.csv", index=False)

In [53]:
query_file = "../data/REV1-SCOPUS-9/export.csv"
query_df = pd.read_csv(query_file, encoding="utf-8", header=0)
query_df.head()

,Authors,Author full names,Author(s) ID,Title,Year,Source title,Volume,Issue,Art. No.,Page start,Page end,Cited by,DOI,Link,Document Type,Publication Stage,Open Access,Source,EID
0,Yalçın S.; Yildirim M.; Khan M.A.; Li Y.; Alab...,"Yalçın, Sercan (57206780804); Yildirim, Muhamm...",57206780804; 57211712877; 57222652080; 5721495...,Optimal control strategy to charging and disch...,2026,Energy Reports,15,NaN,109011,NaN,NaN,1,10.1016/j.egyr.2025.109011,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,All Open Access; Gold Open Access; Green Open ...,Scopus,2-s2.0-105026856721
1,Khanra S.; Kukreja V.K.; Bala I.,"Khanra, Subarna (60243955700); Kukreja, Vijay ...",60243955700; 9737071700; 60067693600,Physics-informed neural networks for different...,2026,Neurocomputing,680,NaN,133317,NaN,NaN,0,10.1016/j.neucom.2026.133317,https://www.scopus.com/inward/record.uri?eid=2...,Review,Final,All Open Access; Hybrid Gold Open Access,Scopus,2-s2.0-105033237171
2,Chen K.; Hui J.; Zhao W.,"Chen, Kunhong (57222067782); Hui, Jizhuang (82...",57222067782; 8230626900; 7403943151,SHURH: A novel multi-domain representation fus...,2026,Journal of Manufacturing Systems,86,NaN,NaN,98,119,0,10.1016/j.jmsy.2026.03.001,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,NaN,Scopus,2-s2.0-105032363776
3,Ren W.; Zhou J.; Qiu X.; Qu X.; Lu Y.; Liu H.,"Ren, Weizhe (58311818200); Zhou, Jiahui (60077...",58311818200; 60077297900; 59666006200; 1504705...,Prediction of tension leg platform motion resp...,2026,Ocean Engineering,351,NaN,124502,NaN,NaN,0,10.1016/j.oceaneng.2026.124502,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,NaN,Scopus,2-s2.0-105030226446
4,Souza M.P.D.O.; Volpatto D.T.; Neto A.M.B.; da...,"Souza, Maurício Prado de Omena (60231595400); ...",60231595400; 57221142720; 58609348800; 5721270...,Modeling flash points of biofuels using thermo...,2026,Fluid Phase Equilibria,605,NaN,114673,NaN,NaN,0,10.1016/j.fluid.2026.114673,https://www.scopus.com/inward/record.uri?eid=2...,Article,Final,All Open Access; Hybrid Gold Open Access,Scopus,2-s2.0-105028030941


In [54]:
# Compute percentage of DOIs in query_df that are in ref_doi
query_doi = query_df["DOI"]
matching_doi = set(query_doi) & ref_doi
print(f"Total DOIs in query: {len(query_doi)}")
print(f"Matching DOIs: {len(matching_doi)}")
print(f"Percentage of matching DOIs: {len(matching_doi) / len(ref_doi) * 100:.2f}%")

Total DOIs in query: 17346
Matching DOIs: 122
Percentage of matching DOIs: 45.52%


In [55]:
# Get the titles of the matching DOIs and not matching DOIs
matching = query_df[query_df["DOI"].isin(matching_doi)]
not_matching = query_df[~query_df["DOI"].isin(matching_doi)]
missing = ref_doi - set(query_doi)

# Query the titles of the matching DOIs
matching_titles = matching["Title"].tolist()
not_matching_titles = not_matching["Title"].tolist()

# get those from the ref_doi_df
ref_missing = ref_doi_df[ref_doi_df["DOI"].isin(missing)]["Title"].tolist()


# Save json for each
import json
from pathlib import Path
folder = Path(query_file).parent
path_matching = folder / "matching_titles.json"
path_not_matching = folder / "not_matching_titles.json"
path_missing = folder / "missing_titles.json"

with open(path_matching, "w", encoding="utf-8") as f:
    json.dump(matching_titles, f, ensure_ascii=False, indent=4)

with open(path_not_matching, "w", encoding="utf-8") as f:
    json.dump(not_matching_titles, f, ensure_ascii=False, indent=4)

with open(path_missing, "w", encoding="utf-8") as f:
    json.dump(list(ref_missing), f, ensure_ascii=False, indent=4)